In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix, classification_report
import numpy as np
import pandas as pd

In [ ]:
#READING IN DATA

data = pd.read_csv("dataset.csv")
data.dtypes

In [ ]:
#VARIABLE EXTRACTION

#x-variables to use
x_vars = ['age', 'job', 'marital', 'education', 'housing', 'month', 'day_of_week', 'campaign', 'pdays']

#extracting x-variables
X = data[x_vars].copy()

#extracting y-variable
y = data['y']

#checking dtypes of x-variables
X.dtypes

In [ ]:
#DATA CLEANING

#making 'unknown' a category on its own
X['housing'] = X['housing'].replace(-1, 2)

#categorical x-variables currently as string type
cat_x_vars = ['job', 'marital', 'education', 'contact', 'month', 'day_of_week']

#one-hot encoding
#rational over label encoding: these x-vars are nominal with no natural order. avoids introducing artificial ordinal relationships
preprocessor = ColumnTransformer(
    transformers=[('cat', OneHotEncoder(handle_unknown='ignore'), #preventing errors on unseen cats during testing
                   cat_x_vars)], 
    remainder='passthrough')

### Decision Tree (baseline model)

In [ ]:
#BUILDING PIPELINE

#pipeline for chaining preproccessing + model
dt_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', DecisionTreeClassifier(random_state=42, class_weight='balanced'))]) #set seed

In [ ]:
#TRAIN/TEST SPLIT

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
#stratify=y makes sure train/test split keeps same proportion of classes as o.g. data

In [ ]:
#MODEL TRAINING & HYPERPARAMETER TUNING (using GridSearchCV)

param_grid = {
    'classifier__max_depth': [5, 6, 7, 8, 9, 10],
    'classifier__min_samples_split': [2, 5, 10],
    'classifier__min_samples_leaf': [1, 2, 4],
    'classifier__criterion': ['gini', 'entropy']
}

grid_search = GridSearchCV(
    dt_pipeline,
    param_grid,
    cv=5,             # 5-fold cross-validation
    scoring='f1',     # evaluation metric
    n_jobs=-1         # use all CPU cores
)
grid_search.fit(X_train, y_train)

print("Best params:", grid_search.best_params_)
print("Best CV F1:", grid_search.best_score_)

In [ ]:
#TESTING MODEL WITH TEST DATA
y_pred = grid_search.predict(X_test)

In [ ]:
#MODEL PERFORMANCE EVALUATION

f1 = f1_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

print("F1-score:", f1)
print("Precision:", precision)
print("Recall:", recall)
print("\nConfusion Matrix:\n", cm)
print("\nClassification Report:\n", classification_report(y_test, y_pred))

### XGBoost Model

In [ ]:
from xgboost import XGBClassifier

In [ ]:
# Handle class imbalance: calculate scale_pos_weight
neg = sum(y_train==0)
pos = sum(y_train==1)
scale_pos_weight = neg / pos

# Initialize pipeline
xgb_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', XGBClassifier(
        use_label_encoder=False, 
        eval_metric='logloss', 
        random_state=42,
        scale_pos_weight=scale_pos_weight
    ))
])

# Parameter grid
param_grid = {
    'classifier__n_estimators': [100, 200, 300],
    'classifier__max_depth': [3, 4, 5, 6],
    'classifier__learning_rate': [0.01, 0.1, 0.2],
    'classifier__subsample': [0.8, 1],
    'classifier__colsample_bytree': [0.8, 1]
}

grid_search_xgb = GridSearchCV(
    xgb_pipeline,
    param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1
)

# Fit model
grid_search_xgb.fit(X_train, y_train)

print("Best params:", grid_search_xgb.best_params_)
print("Best CV F1:", grid_search_xgb.best_score_)

In [ ]:
# Predictions
y_pred = grid_search_xgb.predict(X_test)

# Evaluation
print("F1-score:", f1_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))